# TRIAGE on Kaggle T4×2 (free tier)

Full local pipeline, zero LLM API keys: generator `llama3.1:8b` + judge `qwen2.5:14b`, both via ollama.

**Before running: notebook Settings →**
- Accelerator: **GPU T4 x2**
- Internet: **ON** (requires phone-verified Kaggle account)

Session budget: 12 h wall clock, ~29 GB system RAM. `prepare --rebuild` is the long pole (2–5 h on 4 vCPU + T4 embedding). The benchmark checkpoints per question, so an interrupted run resumes safely.

**Judge options (edit `JUDGE_OLLAMA_MODEL` in cell 4):**
- `qwen2.5:14b` (default, recommended): 8b + 14b ≈ 13.7 GB — both fit one T4, no model swapping. Decent judging quality.
- `qwen2.5:7b`: faster pilot-grade fallback if VRAM gets tight.
- `qwen2.5:32b`: does NOT fit a 16 GB T4 (≈19 GB Q4); ollama CPU-offloads and judging crawls (~3–6 tok/s). Not recommended on this hardware — use the A100/Colab Pro or the `JUDGE_PROVIDER=huggingface` cloud hybrid for 32B+ quality.

In [ ]:
# 1. Clone the repo (full file hierarchy lands on the VM disk)
!git clone --depth 1 https://github.com/Nirajd071/TRIAGE.git
%cd /kaggle/working/TRIAGE
!ls src/

In [ ]:
# 2. Install dependencies (10-15 min)
!pip install -q -r requirements.txt
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz
!pip install -q fastapi uvicorn httpx

In [ ]:
# 3. ollama: install, start server in background, pull models (~14 GB download)
!curl -fsSL https://ollama.com/install.sh | sh
import os
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'   # keep both models resident in VRAM
get_ipython().system_raw('ollama serve > /tmp/ollama.log 2>&1 &')
!sleep 5 && ollama pull llama3.1:8b && ollama pull qwen2.5:14b
!ollama list

In [ ]:
# 4. Environment file - set your WANDB_API_KEY if you want online telemetry,
#    else it stays local (results JSON + figures).
WANDB_API_KEY = ""   # optional

with open('.env', 'w') as f:
    f.write("LLM_PROVIDER=ollama\n")
    f.write("JUDGE_OLLAMA_MODEL=qwen2.5:14b\n")
    f.write("WANDB_MODE=disabled\n" if not WANDB_API_KEY else f"WANDB_API_KEY={WANDB_API_KEY}\n")
print(open('.env').read())

### Optional: restore artifacts from a previous session

Skip this on a first run. After saving artifacts (bottom of this notebook) as a private Kaggle dataset, add that dataset in the notebook's right panel ("+ Add Input"), then run the cell to skip the 2–5 h `prepare --rebuild`.

In [ ]:
# 4b. RESTORE (only if you added your 'triage-artifacts' dataset as Input)
RESTORE = False   # set True after your first successful prepare
if RESTORE:
    !mkdir -p data/processed data/vectordb
    !cp -r /kaggle/input/triage-artifacts/* data/processed/ 2>/dev/null || true
    !cp -r /kaggle/input/triage-artifacts/vectordb/* data/vectordb/ 2>/dev/null || true
    !ls -la data/processed/ data/vectordb/

In [ ]:
# 5. Quick sanity: the API-server unit tests (no GPU, ~5 s)
!python -m pytest tests/test_api_server.py tests/test_judge_provider.py -q

In [ ]:
# 6. Build the knowledge graph + vector store (THE LONG CELL: 2-5 h)
#    Downloads PrimeKG (~1 GB) on first run. Watch RAM: ~29 GB available.
#    If it OOMs, Kernel -> Restart and re-run from cell 1, then use RESTORE.
!python src/main.py prepare --rebuild

In [ ]:
# 7. Smoke test: one query through all three systems
!python src/main.py demo

In [ ]:
# 8. Pilot benchmark: 10 samples (~1 h). Verify results before the full run.
!python src/main.py base-paper --samples 10 --plot

In [ ]:
# 9. Full benchmark: 50 samples (several hours; resumes if interrupted)
!python src/main.py base-paper --samples 50 --plot

In [ ]:
# 10. Inspect the results
import json
r = json.load(open('results/base_paper_benchmark_results.json'))
print(json.dumps({k: r[k] for k in ('scoring_based_assessment', 'retrieval_quality',
      'citation_precision', 'significance_testing')}, indent=2)[:2000])

In [ ]:
# 11. Optional: live API check (the FastAPI server from src/api/server.py)
get_ipython().system_raw('nohup python src/main.py serve --host 127.0.0.1 --port 8000 > /tmp/serve.log 2>&1 &')
!sleep 8 && curl -s http://127.0.0.1:8000/api/triage/health
!curl -s -X POST http://127.0.0.1:8000/api/triage/query -H 'Content-Type: application/json' \
  -d '{"query":"Can Metformin be given to patients with severe renal failure given lactic acidosis risk?","config":{"tauEntropy":0.25}}' | head -c 600
!pkill -f "main.py serve"

In [ ]:
# 12. SAVE artifacts so the next session skips prepare (cells 4b + this)
#    /kaggle/working files become notebook output; download them and upload
#    as a PRIVATE Kaggle dataset named 'triage-artifacts', then use RESTORE=True.
!tar czf /kaggle/working/triage_artifacts.tar.gz \
    data/processed/primekg_graph.gpickle data/processed/sapbert_node_index.faiss \
    data/processed/sapbert_node_names.json data/vectordb
!cp results/base_paper_benchmark_results.json /kaggle/working/ 2>/dev/null
!du -sh /kaggle/working/triage_artifacts.tar.gz
# NOTE: /kaggle/working output limit is ~20 GB; the tarball is a few GB.

## Troubleshooting

- **OOM in cell 6:** Kaggle gives ~29 GB RAM; the PrimeKG graph build is the peak. If it dies, there is no swap — restart and restore from the artifacts dataset instead of re-running prepare.
- **Judge model:** default `qwen2.5:14b` fits alongside the 8b generator on one T4. `qwen2.5:7b` if tight; `qwen2.5:32b` does not fit a T4 (heavy CPU offload, very slow) — for 32B+ quality use the `JUDGE_PROVIDER=huggingface` cloud hybrid instead.
- **ollama dead after restart:** re-run cell 3 (the server doesn't survive kernel restarts).
- **Checkpoint reuse:** re-running cell 9 with the same sample count skips already-evaluated questions and aggregates over the full set.
- **12 h limit:** cell 6 + cell 9 together can exceed it on a first run. Do prepare + pilot in session 1 (save artifacts, cell 12), full benchmark in session 2 (restore first, cell 4b).